<a href="https://colab.research.google.com/github/SANGHATI23/neuroimaging-fhir-omop-fidelity/blob/main/Phase51.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ICHI 2027 · Phase 51 — deep wide-schema audit for longitudinal neuroimaging source evidence

Phase 50 identified candidate real neuroimaging derivative tables, including a selected 258-row / 258-person table, but the first-pass parser did not establish baseline/12-month pairing or bilateral hippocampal measurements.

Phase 51 tests whether those apparent absences are genuine or whether timepoint, laterality, anatomy, and measurement semantics are encoded directly in **wide column names**.

### Phase 51 rules

- Header/schema interpretation is allowed even when source authorization attestation is still pending.
- Row-level aggregate eligibility is executed only when the private authorization attestation is valid.
- No participant identifiers, image identifiers, private paths, or row-level source values leave the private Drive workspace.
- Units are not inferred merely because a field contains the word `volume`; only explicit unit evidence is accepted.
- A wide baseline/12-month field pair is not treated as clinically valid MCI evidence unless a diagnostic/source-group field actually supports that interpretation.


In [1]:
# CODE CELL 1/8 — Drive, constants, helpers
import os, re, csv, json, gzip, hashlib, tempfile, shutil, zipfile, time, math
import datetime as dt
from pathlib import Path
from collections import defaultdict, Counter

PROJECT='ICHI2027_neuroimaging_fhir_omop_fidelity'
PHASE='51_deep_wide_schema_audit_longitudinal_neuroimaging'
EXPECTED_P50_SHAREABLE='d195f42e49e7b07d8d60408e56e636e097ee40af7d5d5261733ce3b715591078'
SELECTED_TOKEN_EXPECTED='069caf1537e082c6'

SUPPORTED={'.csv','.tsv','.txt','.gz','.parquet','.xlsx','.xls'}
SCAN_LIMIT_SECONDS=int(os.environ.get('PHASE51_SCAN_LIMIT_SECONDS','150'))
MAX_FS_ENTRIES=int(os.environ.get('PHASE51_MAX_FS_ENTRIES','30000'))
MAX_SCAN_DEPTH=int(os.environ.get('PHASE51_MAX_SCAN_DEPTH','6'))

def sha256(p):
    h=hashlib.sha256()
    with Path(p).open('rb') as f:
        for b in iter(lambda:f.read(2*1024*1024),b''):
            h.update(b)
    return h.hexdigest()

def token_for_path(p):
    return hashlib.sha256(str(Path(p).resolve()).encode('utf-8')).hexdigest()[:16]

def write_csv(path,rows,fieldnames=None):
    rows=list(rows)
    if fieldnames is None:
        if not rows: raise ValueError('fieldnames required for empty CSV')
        fieldnames=list(rows[0].keys())
    with Path(path).open('w',newline='',encoding='utf-8') as f:
        w=csv.DictWriter(f,fieldnames=fieldnames)
        w.writeheader();w.writerows(rows)

if os.environ.get('PHASE51_LOCAL_DRIVE_ROOT'):
    MYDRIVE=Path(os.environ['PHASE51_LOCAL_DRIVE_ROOT']).expanduser().resolve()
    ROOTS=[MYDRIVE]
else:
    from google.colab import drive
    drive.mount('/content/drive',force_remount=False)
    MYDRIVE=Path('/content/drive/MyDrive')
    assert MYDRIVE.is_dir()
    ROOTS=[MYDRIVE]
    sd=Path('/content/drive/Shareddrives')
    if sd.is_dir():ROOTS.append(sd)

hint=os.environ.get('PHASE51_DRIVE_FOLDER_HINT','').strip()
if hint:
    hp=Path(hint).expanduser()
    if not hp.is_absolute():hp=MYDRIVE/hp
    if not hp.is_dir():raise FileNotFoundError(hp)
    ROOTS=[hp.resolve()]

WORK=Path(tempfile.mkdtemp(prefix='ichi2027_phase51_'))
STAGE=WORK/'outputs';STAGE.mkdir()
OUT_BASE=Path(os.environ.get('PHASE51_OUTPUT_DIR',str(MYDRIVE/'ICHI2027'/'Phase51')))
OUT=OUT_BASE/dt.datetime.now(dt.timezone.utc).strftime('run_%Y%m%dT%H%M%S_%fZ')
PRIVATE=MYDRIVE/'ICHI2027'/'_PRIVATE_REAL_MRI'
PRIVATE.mkdir(parents=True,exist_ok=True)

print('Phase 51 output:',OUT)


Mounted at /content/drive
Phase 51 output: /content/drive/MyDrive/ICHI2027/Phase51/run_20261001T050717_532267Z


In [2]:
# CODE CELL 2/8 — Verify Phase50 and private authorization state

def name_matches(actual,stem,ext):
    return re.fullmatch(re.escape(stem)+r'(?:-\d+|\s*\(\d+\))?'+re.escape(ext),actual,re.I) is not None

p50_candidates=[]
for root in ROOTS:
    for here,dirs,files in os.walk(root,topdown=True,followlinks=False):
        dirs[:]=[d for d in dirs if d not in {'.git','node_modules','.ipynb_checkpoints'} and not d.startswith('.Trash')]
        for fn in files:
            if name_matches(fn,'ICHI2027_Phase50_SHAREABLE','.json'):
                p=Path(here)/fn
                try:
                    if sha256(p)==EXPECTED_P50_SHAREABLE:p50_candidates.append(p)
                except OSError:pass

if not p50_candidates:
    raise FileNotFoundError('Verified Phase50 SHAREABLE.json not found in Drive.')

P50=sorted(p50_candidates,key=str)[0]
p50=json.loads(P50.read_text('utf-8'))
assert p50['phase']=='50_real_mri_derivative_source_inventory_and_eligibility_freeze'
assert p50['selected_source_token']==SELECTED_TOKEN_EXPECTED
assert p50['candidate_source_tables_found']>=1

ATTEST=PRIVATE/'AUTHORIZED_SOURCE_ATTESTATION.json'
ATTEST_TEMPLATE=PRIVATE/'AUTHORIZED_SOURCE_ATTESTATION_TEMPLATE.json'

if not ATTEST_TEMPLATE.exists():
    ATTEST_TEMPLATE.write_text(json.dumps({
      'source_family':'ADNI_or_other_authorized_neuroimaging_derivative',
      'authorized_access':False,
      'authorized_for_local_research_analysis':False,
      'data_use_terms_reviewed':False,
      'participant_level_public_release_prohibited_or_restricted':True,
      'attestation_date_utc':'YYYY-MM-DD',
      'attested_by':'',
      'source_description':'',
      'notes':'Local investigator attestation only; not a substitute for the data-use agreement.'
    },indent=2)+'\n',encoding='utf-8')

attestation_valid=False
if ATTEST.exists():
    try:
        a=json.loads(ATTEST.read_text('utf-8'))
        d=dt.date.fromisoformat(str(a['attestation_date_utc']))
        attestation_valid=(
          d<=dt.datetime.now(dt.timezone.utc).date()
          and a.get('authorized_access') is True
          and a.get('authorized_for_local_research_analysis') is True
          and a.get('data_use_terms_reviewed') is True
          and bool(str(a.get('attested_by','')).strip())
          and bool(str(a.get('source_description','')).strip())
        )
    except Exception:
        attestation_valid=False

print('PASS: Phase50 verified.')
print('Selected source token:',SELECTED_TOKEN_EXPECTED)
print('Authorization attestation valid:',attestation_valid)


PASS: Phase50 verified.
Selected source token: 069caf1537e082c6
Authorization attestation valid: False


In [3]:
# CODE CELL 3/8 — Re-locate the selected private table by token with a bounded filename-first scan

PATH_HINTS=('adni','mri','neuro','freesurfer','free_surfer','ucsf','trace','roi','volume','hippoc','entorh')
priority_terms=('adni','mri','neuro','research','data','trace','freesurfer','project','ichi')

search_starts=[]
for root in ROOTS:
    try:top=[p for p in root.iterdir() if p.is_dir()]
    except Exception:top=[]
    search_starts.extend([p for p in top if any(t in p.name.lower() for t in priority_terms)])
    search_starts.append(root)

start=time.monotonic()
entries=0
selected_path=None
candidate_paths=[]
seen=set()
stack=[(Path(p),0) for p in reversed(search_starts)]

while stack:
    if time.monotonic()-start>SCAN_LIMIT_SECONDS or entries>=MAX_FS_ENTRIES:break
    here,depth=stack.pop()
    try:key=str(here.resolve())
    except:key=str(here)
    if key in seen:continue
    seen.add(key)
    low=str(here).lower()
    if re.search(r'[/\\]ichi2027[/\\]phase\d+',low):continue
    if any(x in low for x in ['/.trash','\\.trash','/.git','\\.git','node_modules','.ipynb_checkpoints']):continue
    try:
        with os.scandir(here) as it:
            for ent in it:
                entries+=1
                if time.monotonic()-start>SCAN_LIMIT_SECONDS or entries>=MAX_FS_ENTRIES:break
                try:
                    if ent.is_dir(follow_symlinks=False):
                        if depth<MAX_SCAN_DEPTH:stack.append((Path(ent.path),depth+1))
                        continue
                    if not ent.is_file(follow_symlinks=False):continue
                except OSError:continue
                p=Path(ent.path)
                if p.suffix.lower() not in SUPPORTED:continue
                plow=(p.name+' '+str(p.parent)).lower()
                if not any(h in plow for h in PATH_HINTS):continue
                tok=token_for_path(p)
                if tok==SELECTED_TOKEN_EXPECTED:
                    selected_path=p
                    break
                candidate_paths.append((tok,p))
        if selected_path is not None:break
    except (PermissionError,OSError):
        continue

if selected_path is None:
    raise FileNotFoundError(
      'Could not re-locate the Phase50 selected source within the bounded scan. '
      'Set PHASE51_DRIVE_FOLDER_HINT to the parent research-data folder and rerun.'
    )

print('Selected private table re-located by non-identifying token.')
print('Filesystem entries inspected:',entries)
print('Scan seconds:',round(time.monotonic()-start,2))


Selected private table re-located by non-identifying token.
Filesystem entries inspected: 2477
Scan seconds: 1.13


In [4]:
# CODE CELL 4/8 — Deep wide-column semantic parser

def normalize(c):
    s=str(c).strip().lower()
    s=s.replace('³','3').replace('^3','3')
    s=re.sub(r'[^a-z0-9]+','_',s).strip('_')
    return s

def anatomy(n):
    if 'hippoc' in n:return 'hippocampus'
    if 'entorh' in n:return 'entorhinal'
    if 'middle_temporal' in n or 'middletemp' in n or 'midtemporal' in n or 'midtemp' in n:return 'middle_temporal'
    if 'ventric' in n:return 'ventricle'
    if re.search(r'(^|_)icv($|_)',n) or 'intracranial' in n or 'etiv' in n:return 'icv'
    return ''

def laterality(n):
    if re.search(r'(^|_)(left|lh)($|_)',n) or n.startswith('lh_') or '_lh_' in n:return 'L'
    if re.search(r'(^|_)(right|rh)($|_)',n) or n.startswith('rh_') or '_rh_' in n:return 'R'
    # FreeSurfer-like compact prefixes/suffixes
    if re.search(r'(^|_)l(hipp|entorh|middle|vent)',n):return 'L'
    if re.search(r'(^|_)r(hipp|entorh|middle|vent)',n):return 'R'
    return ''

def timepoint(n):
    patterns_bl=[
      r'(^|_)(bl|baseline|m0|m00|month0|month_0|v0|visit0|visit_0)($|_)',
      r'(^|_)(screening_baseline)($|_)'
    ]
    patterns_m12=[
      r'(^|_)(m12|month12|month_12|12m|12_month|12months|v12|visit12|visit_12|yr1|year1|year_1)($|_)'
    ]
    if any(re.search(p,n) for p in patterns_bl):return 'BASELINE'
    if any(re.search(p,n) for p in patterns_m12):return 'M12'
    return ''

def unit_evidence(n):
    if re.search(r'(^|_)mm3($|_)',n) or 'cubic_mm' in n:return 'mm3'
    if re.search(r'(^|_)cm3($|_)',n) or 'cubic_cm' in n:return 'cm3'
    return ''

def header_from_selected(p):
    suf=p.suffix.lower()
    if suf=='.gz':
        with gzip.open(p,'rt',encoding='utf-8-sig',errors='replace') as f:
            line=f.readline()
            sep='\t' if line.count('\t')>line.count(',') else ','
            return next(csv.reader([line],delimiter=sep))
    if suf in {'.csv','.tsv','.txt'}:
        with p.open('r',encoding='utf-8-sig',errors='replace') as f:
            line=f.readline()
            sep='\t' if (suf=='.tsv' or line.count('\t')>line.count(',')) else ','
            return next(csv.reader([line],delimiter=sep))
    if suf=='.parquet':
        import pyarrow.parquet as pq
        return pq.ParquetFile(p).schema.names
    if suf in {'.xlsx','.xls'}:
        import pandas as pd
        return list(pd.read_excel(p,nrows=0).columns)
    return []

cols=header_from_selected(selected_path)
parsed=[]
for c in cols:
    n=normalize(c)
    parsed.append({
      'column_token':hashlib.sha256(str(c).encode('utf-8')).hexdigest()[:12],
      'anatomy':anatomy(n),
      'laterality':laterality(n),
      'timepoint':timepoint(n),
      'explicit_unit_in_name':unit_evidence(n),
      'person_key_hint':bool(re.search(r'(^|_)(rid|ptid|subject|participant|person_id|patient_id)($|_)',n)),
      'diagnosis_hint':bool(re.search(r'(^|_)(dx|diagnosis|diagnostic|group|research_group)($|_)',n)),
      'acquisition_id_hint':any(x in n for x in ['studyuid','study_uid','seriesuid','series_uid','imageuid','image_uid','scan_id','image_id']),
      'model_provenance_hint':any(x in n for x in ['freesurfer_version','software_version','model_version','algorithm_version','pipeline_version','provenance','device']),
      'public_original_column_name_exported':False
    })

feat=[r for r in parsed if r['anatomy']]
combos=Counter((r['anatomy'],r['laterality'],r['timepoint']) for r in feat)

coverage=[]
for a in ['hippocampus','entorhinal','middle_temporal','ventricle','icv']:
    for side in (['L','R'] if a!='icv' else ['']):
        for tp in ['BASELINE','M12']:
            coverage.append({
              'anatomy':a,'laterality':side,'timepoint':tp,
              'column_count':combos[(a,side,tp)],
              'present':combos[(a,side,tp)]>0
            })

hip_required=[('hippocampus','L','BASELINE'),('hippocampus','R','BASELINE'),
              ('hippocampus','L','M12'),('hippocampus','R','M12')]
wide_bilateral_hip_longitudinal=all(combos[k]>0 for k in hip_required)

header_summary={
 'selected_file_token':SELECTED_TOKEN_EXPECTED,
 'column_count':len(cols),
 'anatomy_feature_columns':len(feat),
 'baseline_encoded_columns':sum(r['timepoint']=='BASELINE' for r in parsed),
 'm12_encoded_columns':sum(r['timepoint']=='M12' for r in parsed),
 'left_encoded_columns':sum(r['laterality']=='L' for r in parsed),
 'right_encoded_columns':sum(r['laterality']=='R' for r in parsed),
 'hippocampus_columns':sum(r['anatomy']=='hippocampus' for r in parsed),
 'entorhinal_columns':sum(r['anatomy']=='entorhinal' for r in parsed),
 'middle_temporal_columns':sum(r['anatomy']=='middle_temporal' for r in parsed),
 'ventricle_columns':sum(r['anatomy']=='ventricle' for r in parsed),
 'icv_columns':sum(r['anatomy']=='icv' for r in parsed),
 'explicit_mm3_or_cm3_columns':sum(bool(r['explicit_unit_in_name']) for r in parsed),
 'diagnosis_hint_columns':sum(r['diagnosis_hint'] for r in parsed),
 'acquisition_id_hint_columns':sum(r['acquisition_id_hint'] for r in parsed),
 'model_provenance_hint_columns':sum(r['model_provenance_hint'] for r in parsed),
 'wide_bilateral_hippocampus_baseline_m12_established':wide_bilateral_hip_longitudinal
}

print('Deep header audit complete.')
for k,v in header_summary.items():
    if k!='selected_file_token':print(k,':',v)


Deep header audit complete.
column_count : 66
anatomy_feature_columns : 56
baseline_encoded_columns : 8
m12_encoded_columns : 8
left_encoded_columns : 14
right_encoded_columns : 14
hippocampus_columns : 14
entorhinal_columns : 14
middle_temporal_columns : 14
ventricle_columns : 7
icv_columns : 7
explicit_mm3_or_cm3_columns : 0
diagnosis_hint_columns : 0
acquisition_id_hint_columns : 0
model_provenance_hint_columns : 0
wide_bilateral_hippocampus_baseline_m12_established : True


In [5]:
# CODE CELL 5/8 — Determine whether Phase50 missed a wide longitudinal schema

phase50_profile=p50['selected_source_profile']

reinterpretation=[
 {
  'question':'Did Phase50 establish baseline/12-month pairing?',
  'phase50_answer':phase50_profile['paired_baseline_m12_persons']>0,
  'phase51_header_answer':header_summary['baseline_encoded_columns']>0 and header_summary['m12_encoded_columns']>0,
  'interpretation':'Wide timepoint encoding detected in headers' if header_summary['baseline_encoded_columns']>0 and header_summary['m12_encoded_columns']>0 else 'No baseline/M12 header encoding established'
 },
 {
  'question':'Did Phase50 establish bilateral hippocampal measures?',
  'phase50_answer':bool(phase50_profile['bilateral_hippocampus_columns']),
  'phase51_header_answer':wide_bilateral_hip_longitudinal,
  'interpretation':'Wide bilateral hippocampal baseline/M12 fields detected' if wide_bilateral_hip_longitudinal else 'Required L/R baseline/M12 hippocampal field set not established'
 },
 {
  'question':'Are units explicit?',
  'phase50_answer':bool(phase50_profile['explicit_unit_field_present']),
  'phase51_header_answer':header_summary['explicit_mm3_or_cm3_columns']>0,
  'interpretation':'Explicit unit token detected in field name' if header_summary['explicit_mm3_or_cm3_columns']>0 else 'Units still not explicit'
 },
 {
  'question':'Is model/provenance evidence present?',
  'phase50_answer':bool(phase50_profile['model_or_version_field_present']),
  'phase51_header_answer':header_summary['model_provenance_hint_columns']>0,
  'interpretation':'Model/provenance-like header detected' if header_summary['model_provenance_hint_columns']>0 else 'No model/provenance field established'
 }
]

schema_recovered=any((not r['phase50_answer']) and r['phase51_header_answer'] for r in reinterpretation)

print('Phase50 schema reinterpreted:',schema_recovered)
for r in reinterpretation:
    print(r['question'],'=>',r['interpretation'])


Phase50 schema reinterpreted: True
Did Phase50 establish baseline/12-month pairing? => Wide timepoint encoding detected in headers
Did Phase50 establish bilateral hippocampal measures? => Wide bilateral hippocampal baseline/M12 fields detected
Are units explicit? => Units still not explicit
Is model/provenance evidence present? => No model/provenance field established


In [6]:
# CODE CELL 6/8 — Aggregate row-level eligibility only when authorization is valid

aggregate={
 'selected_file_token':SELECTED_TOKEN_EXPECTED,
 'authorization_required':True,
 'authorization_attestation_valid':attestation_valid,
 'row_level_aggregate_audit_executed':False,
 'source_rows':0,
 'unique_persons':0,
 'persons_complete_bilateral_hippocampus_baseline_m12':0,
 'persons_with_mci_like_source_status_and_complete_hippocampus':0,
 'source_ids_exported':False
}

if attestation_valid:
    import pandas as pd
    import numpy as np

    def load_table(p):
        suf=p.suffix.lower()
        if suf=='.gz':return pd.read_csv(p,compression='gzip',low_memory=False)
        if suf=='.csv':return pd.read_csv(p,low_memory=False)
        if suf=='.tsv':return pd.read_csv(p,sep='\t',low_memory=False)
        if suf=='.txt':
            try:return pd.read_csv(p,sep='\t',low_memory=False)
            except:return pd.read_csv(p,low_memory=False)
        if suf=='.parquet':return pd.read_parquet(p)
        if suf in {'.xlsx','.xls'}:return pd.read_excel(p)
        raise ValueError(suf)

    df=load_table(selected_path)
    aggregate['row_level_aggregate_audit_executed']=True
    aggregate['source_rows']=len(df)

    # Resolve person key privately.
    person_cols=[c for c in cols if re.search(r'(^|_)(rid|ptid|subject|participant|person_id|patient_id)($|_)',normalize(c))]
    person=person_cols[0] if person_cols else None
    if person:
        aggregate['unique_persons']=df[person].dropna().astype(str).nunique()

    # Identify required four wide hippocampal columns.
    required={}
    for c in cols:
        n=normalize(c)
        key=(anatomy(n),laterality(n),timepoint(n))
        if key in hip_required:
            required.setdefault(key,[]).append(c)

    if person and all(k in required for k in hip_required):
        masks=[]
        for k in hip_required:
            group=required[k]
            masks.append(df[group].notna().any(axis=1))
        complete=masks[0]
        for m in masks[1:]:complete=complete & m
        complete_people=set(df.loc[complete,person].dropna().astype(str))
        aggregate['persons_complete_bilateral_hippocampus_baseline_m12']=len(complete_people)

        dxcols=[c for c in cols if re.search(r'(^|_)(dx|diagnosis|diagnostic|group|research_group)($|_)',normalize(c))]
        if dxcols:
            mci_mask=False
            for c in dxcols:
                s=df[c].astype(str).str.lower()
                this=s.str.contains(r'\bmci\b|mild cognitive',regex=True,na=False)
                mci_mask=this if isinstance(mci_mask,bool) else (mci_mask|this)
            mci_people=set(df.loc[mci_mask,person].dropna().astype(str))
            aggregate['persons_with_mci_like_source_status_and_complete_hippocampus']=len(complete_people & mci_people)

print('Authorization valid:',attestation_valid)
print('Row-level aggregate audit executed:',aggregate['row_level_aggregate_audit_executed'])
if attestation_valid:
    print('Complete bilateral hippocampus baseline/M12 persons:',aggregate['persons_complete_bilateral_hippocampus_baseline_m12'])


Authorization valid: False
Row-level aggregate audit executed: False


In [7]:
# CODE CELL 7/8 — Readiness decision after deep schema audit

gates=[
 {'gate':'selected_real_derivative_source_relocated','passed':selected_path is not None},
 {'gate':'authorization_attestation_valid','passed':attestation_valid},
 {'gate':'baseline_columns_encoded_or_longitudinal_pairing_established',
  'passed':header_summary['baseline_encoded_columns']>0 and header_summary['m12_encoded_columns']>0},
 {'gate':'bilateral_hippocampus_baseline_m12_schema_established',
  'passed':wide_bilateral_hip_longitudinal},
 {'gate':'explicit_unit_evidence_present',
  'passed':header_summary['explicit_mm3_or_cm3_columns']>0},
 {'gate':'acquisition_or_image_identifier_field_present',
  'passed':header_summary['acquisition_id_hint_columns']>0},
 {'gate':'model_or_provenance_field_present',
  'passed':header_summary['model_provenance_hint_columns']>0},
 {'gate':'diagnostic_source_field_present',
  'passed':header_summary['diagnosis_hint_columns']>0},
 {'gate':'authorized_row_level_complete_longitudinal_hippocampus_persons_gt0',
  'passed':bool(attestation_valid and aggregate['persons_complete_bilateral_hippocampus_baseline_m12']>0)}
]

passed=sum(int(g['passed']) for g in gates)

if not attestation_valid:
    next_step='BLOCKED_PENDING_LOCAL_AUTHORIZED_USE_ATTESTATION'
elif aggregate['persons_complete_bilateral_hippocampus_baseline_m12']>0:
    next_step='PHASE52_FREEZE_REAL_SOURCE_MANIFEST_AND_APPLY_FROZEN_NEUROIMAGING_METHOD'
elif wide_bilateral_hip_longitudinal:
    next_step='BLOCKED_NO_COMPLETE_PERSON_LEVEL_LONGITUDINAL_HIPPOCAMPUS_AFTER_AUTHORIZED_AUDIT'
else:
    next_step='BLOCKED_REQUIRED_LONGITUDINAL_BILATERAL_HIPPOCAMPUS_SCHEMA_NOT_ESTABLISHED'

print('Phase51 readiness:',passed,'/',len(gates))
print('Next step:',next_step)


Phase51 readiness: 3 / 9
Next step: BLOCKED_PENDING_LOCAL_AUTHORIZED_USE_ATTESTATION


In [8]:
# CODE CELL 8/8 — Public-safe outputs

def save(name,rows,fields=None):
    p=STAGE/name
    write_csv(p,rows,fields)
    return p

artifacts=[]
artifacts.append(save('ICHI2027_Phase51_COLUMN_SEMANTIC_TOKENS.csv',parsed))
artifacts.append(save('ICHI2027_Phase51_WIDE_FEATURE_COVERAGE.csv',coverage))
artifacts.append(save('ICHI2027_Phase51_PHASE50_REINTERPRETATION.csv',reinterpretation))
artifacts.append(save('ICHI2027_Phase51_READINESS_GATES.csv',gates))
artifacts.append(save('ICHI2027_Phase51_AGGREGATE_ELIGIBILITY.csv',[aggregate]))

header_path=STAGE/'ICHI2027_Phase51_HEADER_SUMMARY.json'
header_path.write_text(json.dumps(header_summary,sort_keys=True,indent=2)+'\n',encoding='utf-8')
artifacts.append(header_path)

readme=STAGE/'ICHI2027_Phase51_README.txt'
readme.write_text(
 'Phase51 tests whether the Phase50 selected 258-person source uses a wide schema with anatomy, laterality, and timepoint encoded in column names.\n'
 'Original private column names are not exported; only hashed column tokens and semantic classifications are public.\n'
 'Header/schema interpretation may run while authorization attestation is pending.\n'
 'Row-level aggregate eligibility runs only after the private authorization attestation is valid.\n'
 'Units, MCI status, acquisition IDs, and model/provenance are never invented when absent.\n',
 encoding='utf-8'
)
artifacts.append(readme)

summary={
 'project':PROJECT,
 'phase':PHASE,
 'run_utc':dt.datetime.now(dt.timezone.utc).isoformat(),
 'input_phase50_shareable_sha256':sha256(P50),
 'selected_source_token':SELECTED_TOKEN_EXPECTED,
 'authorization_attestation_valid':attestation_valid,
 'phase50_schema_reinterpreted_by_deep_wide_parser':schema_recovered,
 'header_summary':header_summary,
 'aggregate_eligibility':aggregate,
 'readiness_gates_passed':passed,
 'readiness_gates_total':len(gates),
 'next_step':next_step,
 'real_authorized_mri_processed_by_proposed_method':False,
 'official_omop_etl_executed':False,
 'official_micdm_load_executed':False,
 'independently_blinded_external_validation':False,
 'privacy':'NO_RAW_COLUMNS_NO_SOURCE_IDS_NO_PRIVATE_PATHS_NO_ROW_LEVEL_EXPORT',
 'artifact_sha256':{p.name:sha256(p) for p in artifacts},
 'warning':'Wide-schema recovery establishes source-field availability only. It does not establish authorization, clinical correctness, official target transformation, or real-data method performance.'
}
summary_path=STAGE/'ICHI2027_Phase51_SHAREABLE.json'
summary_path.write_text(json.dumps(summary,sort_keys=True,indent=2)+'\n',encoding='utf-8')
artifacts.append(summary_path)

zip_path=STAGE/'ICHI2027_Phase51_PUBLIC_OUTPUTS.zip'
with zipfile.ZipFile(zip_path,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for p in artifacts:z.write(p,p.name)

OUT.mkdir(parents=True,exist_ok=False)
for p in artifacts+[zip_path]:
    dest=OUT/p.name
    for attempt in range(3):
        try:
            shutil.copyfile(p,dest)
            assert sha256(dest)==sha256(p)
            break
        except Exception as e:
            if attempt==2:raise RuntimeError('Drive save/readback verification failed for '+p.name) from e
            time.sleep(1+attempt)

print('SUCCESS: Phase51 deep wide-schema audit completed.')
print('Schema reinterpreted:',schema_recovered)
print('Readiness:',passed,'/',len(gates))
print('Next step:',next_step)
print('Shareable:',OUT/summary_path.name)
print('Public ZIP:',OUT/zip_path.name)


SUCCESS: Phase51 deep wide-schema audit completed.
Schema reinterpreted: True
Readiness: 3 / 9
Next step: BLOCKED_PENDING_LOCAL_AUTHORIZED_USE_ATTESTATION
Shareable: /content/drive/MyDrive/ICHI2027/Phase51/run_20261001T050717_532267Z/ICHI2027_Phase51_SHAREABLE.json
Public ZIP: /content/drive/MyDrive/ICHI2027/Phase51/run_20261001T050717_532267Z/ICHI2027_Phase51_PUBLIC_OUTPUTS.zip


## What to look for after Phase 51

The most informative outcome is whether Phase 51 changes the Phase 50 interpretation:

- If baseline/M12 and bilateral hippocampal columns are recovered from the wide field names, the source is structurally much closer to the intended real-data experiment than Phase 50 suggested.
- If those fields remain absent, the current selected derivative table is not sufficient for the planned longitudinal hippocampal experiment by itself.
- In either case, row-level real-data analysis remains blocked until the private authorization attestation is valid.
